# Experiment Segment Durations

Per-subject and group-level summary of:
- **Baseline duration** (NREM onset → first stim) in minutes
- **Total experiment duration** (first stim → end of last pause) in minutes
- **Number of stim segments**
- **Average stim-segment duration** in minutes
- **Average pause/break duration** in minutes

Computed separately for the **stimulation group** and the **control group**.

In [1]:
import numpy as np
import pandas as pd
import sys, os

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
import utils

In [2]:
# ============================================================
# Subject lists (final)
# ============================================================
stim_subjects = ['485', '486', '487', '488', '497', '498', '499',
                 '505', '5107', '515', '520', '545']

control_subjects = ['394', '396', '398', '402', '404', '405',
                    '406', '415', '416', '417']


In [3]:
# ============================================================
# Compute segment durations — STIM group
# ============================================================
# Baseline = nrem_epochs[0][0] → stim_epochs[0][0]
# (for 515 / 545 baseline is negative → use 10 min)
# Stim blocks from get_nrem_epochs → stim
# Pauses = gaps between consecutive stim blocks; last pause = 5 min

stim_rows = []
for subj in stim_subjects:
    uid = subj
    nrem_epochs, nrem_stim_epochs, stim_epochs = utils.get_nrem_epochs(uid)

    nrem_start  = nrem_epochs[0][0]
    stim_start  = stim_epochs[0][0]
    stim_end    = stim_epochs[-1][1]

    # Baseline
    baseline_min = (stim_start - nrem_start) / 60
    if baseline_min < 0:
        baseline_min = None   # fallback for 515, 545

    # Stim segment durations
    stim_durs = [(e - s) / 60 for s, e in stim_epochs]

    # Pause durations (gaps between blocks)
    pause_durs = []
    for i in range(len(stim_epochs) - 1):
        gap = (stim_epochs[i + 1][0] - stim_epochs[i][1]) / 60
        pause_durs.append(gap)
    pause_durs.append(5.0)  # last pause = 5 min after last stim

    # Total experiment: first stim → end of last pause
    total_exp = (stim_end + 5 * 60 - stim_start) / 60

    stim_rows.append({
        'subject':              subj,
        'baseline_min':         round(baseline_min, 2) if baseline_min is not None else None,
        'total_experiment_min': round(total_exp, 2),
        'n_stim_segments':      len(stim_epochs),
        'avg_stim_dur_min':     round(np.mean(stim_durs), 2),
        'avg_pause_dur_min':    round(np.mean(pause_durs), 2),
    })

df_stim = pd.DataFrame(stim_rows)
print(f'Stim group: {len(df_stim)} subjects\n')
df_stim

Stim group: 12 subjects


,subject,baseline_min,total_experiment_min,n_stim_segments,avg_stim_dur_min,avg_pause_dur_min
0,485,52.18,52.23,4,5.00,8.06
1,486,43.62,72.06,4,11.82,6.20
2,487,14.99,114.77,6,9.51,9.62
3,488,77.24,91.79,7,5.87,7.24
4,497,16.93,186.06,15,4.64,7.76
5,498,35.63,134.97,12,5.46,5.79
6,499,5.87,89.19,8,5.39,5.76
7,505,106.10,50.05,5,4.89,5.12
8,5107,119.59,50.15,5,4.87,5.16
9,515,NaN,102.75,6,6.37,10.75


In [4]:
# Stim group summary
cols = ['baseline_min', 'total_experiment_min', 'n_stim_segments',
        'avg_stim_dur_min', 'avg_pause_dur_min']

summary_stim = pd.DataFrame({
    'mean': df_stim[cols].mean().round(2),
    'std':  df_stim[cols].std().round(2),
    'min':  df_stim[cols].min().round(2),
    'max':  df_stim[cols].max().round(2),
})
print('=== STIM GROUP SUMMARY ===')
summary_stim

=== STIM GROUP SUMMARY ===


,mean,std,min,max
baseline_min,48.12,40.68,5.87,119.59
total_experiment_min,86.25,43.51,39.90,186.06
n_stim_segments,6.75,3.44,4.00,15.00
avg_stim_dur_min,6.14,2.22,4.64,11.82
avg_pause_dur_min,6.81,1.90,5.12,10.75


In [ ]:
# ============================================================
# Compute segment durations — CONTROL group
# ============================================================
# Baseline = nrem_epochs[0][0] → stim_epochs[0][0]
# (controls skip first 30 min after NREM onset in the analysis,
#  but baseline_minutes is computed as stim_start - nrem_start)
# Stim blocks from get_stim_starts_upgraded (synthetic 5-on/5-off)

ctrl_rows = []
for subj in control_subjects:
    nrem_epochs = utils.get_control_nrem_epochs(subj)
    stim_epochs = utils.get_stim_starts_upgraded(subj)

    if len(stim_epochs) == 0 or len(nrem_epochs) == 0:
        print(f'  ⚠ No data for control {subj}')
        continue

    nrem_start  = nrem_epochs[0][0]
    stim_start  = stim_epochs[0][0]
    stim_end    = stim_epochs[-1][1]

    # Baseline (same logic as stim group)
    baseline_min = (stim_start - nrem_start) / 60
    if baseline_min < 0:
        print(f'  ⚠ Unexpected negative baseline for control {subj}: {baseline_min:.2f} min')
        baseline_min = None  # or raise an error

    # Stim segment durations
    stim_durs = [(e - s) / 60 for s, e in stim_epochs]

    # Pause durations
    pause_durs = []
    for i in range(len(stim_epochs) - 1):
        gap = (stim_epochs[i + 1][0] - stim_epochs[i][1]) / 60
        pause_durs.append(gap)
    pause_durs.append(5.0)

    # Total experiment
    total_exp = (stim_end + 5 * 60 - stim_start) / 60

    ctrl_rows.append({
        'subject':              subj,
        'baseline_min':         round(baseline_min, 2),
        'total_experiment_min': round(total_exp, 2),
        'n_stim_segments':      len(stim_epochs),
        'avg_stim_dur_min':     round(np.mean(stim_durs), 2),
        'avg_pause_dur_min':    round(np.mean(pause_durs), 2),
    })

df_ctrl = pd.DataFrame(ctrl_rows)
print(f'Control group: {len(df_ctrl)} subjects\n')
df_ctrl

In [ ]:
# Control group summary
summary_ctrl = pd.DataFrame({
    'mean': df_ctrl[cols].mean().round(2),
    'std':  df_ctrl[cols].std().round(2),
    'min':  df_ctrl[cols].min().round(2),
    'max':  df_ctrl[cols].max().round(2),
})
print('=== CONTROL GROUP SUMMARY ===')
summary_ctrl

In [ ]:
# ============================================================
# Combined text summary
# ============================================================
def print_summary(label, df, summary):
    print(f'\n{"="*60}')
    print(f'  {label}  (n = {len(df)})')
    print(f'{"="*60}')
    nice_names = {
        'baseline_min':         'Baseline duration (min)',
        'total_experiment_min': 'Total experiment duration (min)',
        'n_stim_segments':      '# stim segments',
        'avg_stim_dur_min':     'Avg stim segment duration (min)',
        'avg_pause_dur_min':    'Avg pause duration (min)',
    }
    for col in cols:
        m = summary.loc[col, 'mean']
        s = summary.loc[col, 'std']
        print(f'  {nice_names[col]:40s}  {m:7.2f} ± {s:.2f}')
    print()

print_summary('STIM GROUP',    df_stim, summary_stim)
print_summary('CONTROL GROUP', df_ctrl,  summary_ctrl)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. הוספת עמודת זיהוי קבוצה ואיחוד הטבלאות
df_stim['Group'] = 'Stim'
df_ctrl['Group'] = 'Control'

# איחוד לשם ציור הגרפים
df_combined = pd.concat([df_stim, df_ctrl], ignore_index=True)

# 2. הגדרת העמודות שנרצה לצייר והכותרות היפות שלהן
cols_to_plot = {
    'baseline_min': 'Baseline Duration (min)',
    'total_experiment_min': 'Total Experiment Duration (min)',
    'n_stim_segments': 'Number of Stim Segments',
    'avg_stim_dur_min': 'Average Stim Duration (min)',
    'avg_pause_dur_min': 'Average Pause Duration (min)'
}

# 3. הגדרת צבעים לקבוצות (אפשר לשנות לפי טעמך)
palette = {'Stim': '#4C72B0', 'Control': '#DD8452'}

# 4. יצירת רשת הגרפים (2 שורות, 3 עמודות)
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
axes = axes.flatten() # הפיכה למערך חד-מימדי כדי שיהיה קל לרוץ עליו בלולאה

# 5. ריצה על כל עמודה וציור הגרף
for i, (col, title) in enumerate(cols_to_plot.items()):
    ax = axes[i]
    
    # ציור ה-Boxplot (ללא נקודות חריגות כדי שלא יופיעו פעמיים בגלל ה-stripplot)
    sns.boxplot(data=df_combined, x='Group', y=col, ax=ax, 
                palette=palette, width=0.4, boxprops=dict(alpha=0.5), showfliers=False)
    
    # ציור הנקודות של כל נבדק (stripplot)
    sns.stripplot(data=df_combined, x='Group', y=col, ax=ax, 
                  palette=palette, size=7, jitter=True, alpha=0.8, edgecolor='black', linewidth=1)
    
    # עיצוב הגרף
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(col)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # הסרת המסגרת העליונה והימנית למראה נקי יותר
    sns.despine(ax=ax)

# מחיקת הגרף השישי (כי יש לנו רק 5 עמודות)
fig.delaxes(axes[5])

# סידור הרווחים בין הגרפים
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

# 1. שליפת נתוני הבייסליין וסינון ערכי חסר (NaN / None)
baseline_stim = df_stim['baseline_min'].dropna()
baseline_ctrl = df_ctrl['baseline_min'].dropna()

# 2. ביצוע המבחן הסטטיסטי (Mann-Whitney U Test)
u_stat, p_val = stats.mannwhitneyu(baseline_stim, baseline_ctrl, alternative='two-sided')

# 3. הדפסת התוצאות בצורה קריאה
print("=== Baseline Length Comparison ===")
print(f"Stim Group:    Median = {baseline_stim.median():.2f} min (n = {len(baseline_stim)})")
print(f"Control Group: Median = {baseline_ctrl.median():.2f} min (n = {len(baseline_ctrl)})")
print("-" * 32)
print(f"U-Statistic: {u_stat}")
print(f"P-value:     {p_val:.4f}")

# בדיקת מובהקות (אלפא = 0.05)
if p_val < 0.05:
    print("\nConclusion: There IS a significant difference in baseline length between the groups.")
else:
    print("\nConclusion: There is NO significant difference in baseline length (n.s.).")